# Example notebook to demonstrate the usage of the Lettuce Greenhouse environment

This notebook shows you how to interact with the lettuce greenhouse model via the **gymnasium** environment wrapper, [see docs](https://gymnasium.farama.org/index.html).

It consists of three parts:

1. Defining the environment
2. Simulating the environment
3. Visualising the environment

Play around a bit to get yourself familiar with the source code of the environment.

In [ ]:
%pip install -r "C:\Users\dylan\OneDrive - Wageningen University & Research\Advanced Machine Learning\LettuceGreenhouse\aml_project\requirements.txt"

In [ ]:
import sys
import os

notebook_dir = os.path.abspath('')

root_dir = os.path.abspath(os.path.join(notebook_dir, '..', '..'))

if root_dir not in sys.path:
    sys.path.append(root_dir)

import numpy as np
from aml_project.environments.lettuce_greenhouse import LettuceGreenhouse
from aml_project.common.helper_functions import weather2ppmrh
from aml_project.common.plot_gh_variables import create_trajectory_figure, plot_trajectory

### 1. Define the environment

Here you define arguments that describe the environment setup. Some are fixed, some could be changed. I suggest you use the defaults for now.

- weather_data_dir (location of the weather data)
- nx (number of states)
- ny (number of measurements)
- nd (number of disturbances)
- nu (number of control variables)
- control rate (interval between control inputs in minutes)
- c (number of seconds in day)
- n_days (number of days you want to run the simulation)
- Np (number future weather predictions to use in the observation space)
- start_day (day of the year you want to start the simulation)
- var_weather (whether to randomize the starting date during training)
- noise (whether to add noise to the weather predictions)
- reward_coefs (reward coefficients, helpful when defining reward function)
- penalty_coefs (penalty coefficients, helpful when defining penalty function)


In [ ]:
env_kargs = {
    "weather_data_dir": "../environments/weather/outdoorWeatherWurGlas2014.mat",
        "nx": 4,                
        "ny": 4,                
        "nd": 4,                
        "nu": 3,                
        "control_rate":  15,    
        "c": 86400,         
        "n_days": 2,        
        "Np": 20,           
        "start_day": 40,    
        "var_weather": False,
        "noise": False,
        "reward_coefs":np.ones(4),
        "penalty_coefs":np.ones(3)
}

# initialise the environment
env = LettuceGreenhouse(**env_kargs)

### 2. Generate a random trajectory

This line of code will random sample actions from the action space of the environment and leaves you with three matrices:

- `y` measurement at each time step
- `d` weather disturbances at each timestep
- `u` control inputs at each timestep

Before running the environment, always make sure you call env.reset(), to reset it to its initial state.

In [ ]:
env.reset()
y, d, u = env.generate_trajectory()

### 3. Plot your generated trajectories

Now you can plot the trajectories for `y, d, u`. Two functions visualise these variables in a single plot. It plots the number of days since the start of the simulation versus the variable value. You can easily add another trajectory with the `plot_trajectory()` function.

In [ ]:
nvars = 11
# plot the resulting trajectory
n_per_day = int(24*3600/env.h) # control frequency per day
label= 'Random policy'
trajectory = np.concatenate((y, d, u), axis=1)
fig, axes = create_trajectory_figure(nvars, env.L, env.h, env.c, None, None)
fig, axes = plot_trajectory(fig, axes, trajectory, env.L, env.h, env.c, 0, env.n_days, n_per_day, label)


In [ ]:
%pip install openpyxl

In [ ]:
import os
import numpy as np
import pandas as pd
from openpyxl import Workbook, load_workbook

# Generate a new random trajectory
env.reset()
y, d, u = env.generate_trajectory()
trajectory = np.concatenate((y, d, u), axis=1)

# Column names for the Excel file
headers = (
    [f"y_{i}" for i in range(y.shape[1])] +
    [f"d_{i}" for i in range(d.shape[1])] +
    [f"u_{i}" for i in range(u.shape[1])]
)

df = pd.DataFrame(trajectory, columns=headers)

file_path = "Example.xlsx"

if os.path.exists(file_path):
    wb = load_workbook(file_path)
    if "Trajectory" not in wb.sheetnames:
        ws = wb.create_sheet("Trajectory")
        ws.append(headers)
    else:
        ws = wb["Trajectory"]
        if ws.max_row == 0:
            ws.append(headers)
    for row in df.itertuples(index=False, name=None):
        ws.append(row)
    wb.save(file_path)
    wb.close()
else:
    wb = Workbook()
    ws = wb.active
    ws.title = "Trajectory"
    ws.append(headers)
    for row in df.itertuples(index=False, name=None):
        ws.append(row)
    wb.save(file_path)
    wb.close()

print(f"Saved to {file_path}")